# HyperParameter Tuning Using KerasTuner

In [162]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [163]:
X,y = load_breast_cancer(return_X_y= True)

In [164]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [165]:
import tensorflow 
from tensorflow import keras
from keras import Sequential 
from keras.layers import Dense
from keras.metrics import R2Score
import kerastuner as kt

In [166]:
def build_model(hp):
    model = Sequential()
    optimizer = hp.Choice('optimizer', values = ['sgd','rmsprop','adam'])

    model.add(Dense(units = hp.Int('unit' + str(0) , min_value = 30, max_value = 120, step = 10), activation = 'relu',input_dim = 30))
    for i in range(hp.Int('hidden_layers',min_value = 3, max_value = 6)):
        model.add(Dense(units = hp.Int('unit' + str(i), min_value = 30, max_value = 120, step = 10) , activation= 'relu'))
    
    model.add(Dense(1, activation= 'sigmoid'))

    model.compile(loss = 'binary_crossentropy', optimizer = optimizer, metrics  = ['accuracy'])

    return model

In [167]:
tuner = kt.RandomSearch(build_model, 
                        objective = 'val_accuracy',
                        max_trials= 5,
                        overwrite = True)

d:\DeepLearning.py\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [168]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test), verbose = 1)

Trial 5 Complete [00h 00m 08s]
val_accuracy: 0.9736841917037964

Best val_accuracy So Far: 0.9912280440330505
Total elapsed time: 00h 00m 40s


In [169]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam',
 'unit0': 110,
 'hidden_layers': 3,
 'unit1': 80,
 'unit2': 100,
 'unit3': 100,
 'unit4': 40,
 'unit5': 120}

In [170]:
model = tuner.get_best_models(num_models= 1)[0]

d:\DeepLearning.py\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(store)


In [172]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 110)            │         3,410 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 110)            │        12,210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 80)             │         8,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │         8,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,701 (127.74 KB)

 Trainable params: 32,701 (127.74 KB)

 Non-trainable params: 0 (0.00 B)

In [173]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5 ).astype(int)
print('R2 Score: ',accuracy_score(y_test, y_pred))


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step 
R2 Score:  0.9912280701754386
